In [8]:
import numpy as np
import wfdb
import torch.optim as optim
import os
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

In [2]:
class ECGDataset(Dataset):
    def __init__(self, data_dir, prefix_dwt='X_dwt_', prefix_stft='X_stft_', prefix_y='y_'):
        self.data_dir = data_dir
        
        # Собираем список всех файлов батчей
        self.dwt_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_dwt)])
        self.stft_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_stft)])
        self.y_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_y)])
        
        self.cumulative_sizes = []
        current_total = 0
        
        # Индексируем датасет (считаем, сколько окон в каждом файле)
        for f in self.y_files:
            labels = np.load(os.path.join(data_dir, f))
            current_total += len(labels)
            self.cumulative_sizes.append(current_total)
        
        self.total_size = current_total
        # Кэш для последнего загруженного файла (чтобы не читать диск на каждом шаге)
        self.current_batch_idx = -1
        self.batch_data = (None, None, None)

    def __len__(self):
        return self.total_size

    def _load_batch_for_index(self, global_idx):
        # Определяем, в каком файле лежит этот индекс
        batch_idx = next(i for i, size in enumerate(self.cumulative_sizes) if size > global_idx)
        
        # Если файл уже в памяти, ничего не делаем
        if batch_idx == self.current_batch_idx:
            return batch_idx
            
        # Загружаем новые данные
        dwt = np.load(os.path.join(self.data_dir, self.dwt_files[batch_idx]))
        stft = np.load(os.path.join(self.data_dir, self.stft_files[batch_idx]))
        y = np.load(os.path.join(self.data_dir, self.y_files[batch_idx]))
        
        self.batch_data = (dwt, stft, y)
        self.current_batch_idx = batch_idx
        return batch_idx

    def __getitem__(self, idx):
        batch_idx = self._load_batch_for_index(idx)
        
        # Вычисляем локальный индекс внутри батча
        start_idx = self.cumulative_sizes[batch_idx - 1] if batch_idx > 0 else 0
        local_idx = idx - start_idx
        
        d_sample, s_sample, y_sample = self.batch_data
        
        # Извлекаем данные
        x_dwt = d_sample[local_idx]   # (125, 120, 8)
        x_stft = s_sample[local_idx]  # (129, 21, 8)
        target = y_sample[local_idx]  # One-hot вектор
        
        # Переставляем оси под PyTorch: (H, W, C) -> (C, H, W)
        x_dwt = torch.from_numpy(x_dwt).permute(2, 0, 1).float()
        x_stft = torch.from_numpy(x_stft).permute(2, 0, 1).float()
        target = torch.from_numpy(target).float()
        
        return x_dwt, x_stft, target


In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class SEAttention(nn.Module):
    """ Простой механизм внимания для взвешивания каналов """
    def __init__(self, channels, reduction=4):
        super(SEAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ECGMultiModalNet(nn.Module):
    def __init__(self, num_classes):
        super(ECGMultiModalNet, self).__init__()
        
        # Ветка DWT
        self.dwt_branch = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=(3, 7), padding=(1, 3)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Ветка STFT
        self.stft_branch = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Общий классификатор
        self.classifier = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_dwt, x_stft):
        feat_dwt = self.dwt_branch(x_dwt)
        feat_stft = self.stft_branch(x_stft)
        
        combined = torch.cat((feat_dwt, feat_stft), dim=1)
        return self.classifier(combined)

In [4]:
def train_model(train_loader, val_loader, device, epochs=10, threshold=0.5, start_epoch=0, path='last_checkpoint'):
    for epoch in range(start_epoch, epochs):
        model.train()
        running_loss = 0.0
        
        all_preds = []
        all_labels = []
        
        for x_dwt, x_stft, labels in train_loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(x_dwt, x_stft)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            # Переводим логиты в вероятности и применяем порог
            preds = (torch.sigmoid(outputs) > threshold).int()
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
        # Склеиваем всё в один массив для расчета метрик
        all_preds = np.vstack(all_preds)
        all_labels = np.vstack(all_labels)
        
        # Считаем F1
        train_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        
        # Валидация
        val_loss, val_f1 = validate(model, val_loader, criterion, device, threshold)
        
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
        }
        torch.save(checkpoint, path+'.pth')

        print(f"Epoch {epoch+1} | LR: {current_lr:.6f}")
        print(f"Train Loss: {running_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        print("-" * 30)

def validate(model, loader, criterion, device, threshold=0.5):
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x_dwt, x_stft, labels in loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            outputs = model(x_dwt, x_stft)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = (torch.sigmoid(outputs) > threshold).int()
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return val_loss/len(loader), val_f1

In [6]:
path_full = "C:\\Users\\HundeRob0t\\.cache\\kagglehub\\datasets\\gamalasran\\physionet-challenge-2020\\versions\\1\\classification-of-12-lead-ecgs-the-physionetcomputing-in-cardiology-challenge-2020-1.0.2\\training\\cpsc_2018_extra\\"
g_list = ["g1\\","g2\\","g3\\","g4\\"]
RECORDS = []
for g in g_list:
    with open (path_full+g+"RECORDS", 'r', encoding='utf-8') as f:
        RECORDS += f.read().split("\n")[:-1]

In [10]:
def onehot_group(target):
    oh = np.zeros(12)
    st_t_codes = {'428750005', '164930006', '164934002', '429622005'} # Изменения ST/T
    rare_imp = {'111975006', '164909002', '195042002', '54329005'} # QT, БЛНПГ, и т.д.

    for code in target:
        # 0. Норма
        if code == '427084000' or code == '164865005': 
            oh[0] = 1
        # 1. Изменения ST-T
        elif code in st_t_codes:
            oh[1] = 1
        # 2. Мерцалка/Трепетание (AF/AFL)
        elif code == '164889003' or code == '164890007':
            oh[2] = 1
        # 3. Брадикардия
        elif code == '164861001':
            oh[3] = 1
        # 4. Желудочковая экстрасистолия (PVC)
        elif code == '427172004':
            oh[4] = 1
        # 5. Предсердная экстрасистолия (PAC)
        elif code == '284470004':
            oh[5] = 1
        # 6. Блокада левой ножки (LBBB)
        elif code == '164909002':
            oh[6] = 1
        # 7. Блокада правой ножки (RBBB)
        elif code == '164867002' or code == '164931005':
            oh[7] = 1
        # 8. Отклонение оси влево (LAD)
        elif code == '164873001':
            oh[8] = 1
        # 9. АВ-блокада 1 степени
        elif code == '270492004':
            oh[9] = 1
        # 10. Тахикардии
        elif code == '426177001' or code == '426761007':
            oh[10] = 1
        # 11. Rare Important (Опасные/Важные редкие)
        elif code in rare_imp:
            oh[11] = 1
            
    return oh

In [11]:
all_code_freqs_gr = np.zeros(12)
for g in g_list:
    for rec_id in RECORDS:
        try:
            rec = wfdb.rdrecord(path_full+g+rec_id)
        except FileNotFoundError:
            continue
        header_rec = rec.comments
        diag = header_rec[2][4:].split(",")
    
        all_code_freqs_gr += onehot_group(diag)
all_code_freqs_gr = all_code_freqs_gr.astype(int)

total_samples = len(RECORDS)
weights = (total_samples - all_code_freqs_gr) / (all_code_freqs_gr + 1e-6)
weights = np.log1p(weights)
weights

array([1.65475886, 0.62513944, 2.8142799 , 2.19635614, 2.91055673,
       3.85653924, 4.50941251, 1.03467125, 3.08440366, 3.48355959,
       4.27579767, 3.68109057])

In [13]:
num_classes = 12
fs = 500

In [ ]:
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используем: {device}")

    model = ECGMultiModalNet(num_classes=num_classes).to(device)

    pos_weights = torch.tensor(weights, dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=4, threshold=0.001, threshold_mode='abs')
    
    train_dataset = ECGDataset(data_dir='D:/processed_data/train')
    val_dataset = ECGDataset(data_dir='D:/processed_data/val')

    train_loader = DataLoader(
        train_dataset, 
        batch_size=32, 
        shuffle=True, # для качества True, для скорости False (связано с чтением файлов и подачей их на обучение)
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False
    )
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    
    # Для запуска обучения
    # train_model(train_loader, val_loader, device, epochs=15)

Обучение (открой ячейку)
<!-- Используем: cuda
Epoch 1 | LR: 0.001000
Train Loss: 0.4081 | Train F1: 0.1570
Val Loss: 0.4029 | Val F1: 0.1571
------------------------------
Epoch 2 | LR: 0.001000
Train Loss: 0.3581 | Train F1: 0.1997
Val Loss: 0.3751 | Val F1: 0.1897
------------------------------
Epoch 3 | LR: 0.001000
Train Loss: 0.3399 | Train F1: 0.2318
Val Loss: 0.4011 | Val F1: 0.1527
------------------------------
Epoch 4 | LR: 0.001000
Train Loss: 0.3244 | Train F1: 0.2702
Val Loss: 0.3705 | Val F1: 0.2243
------------------------------
Epoch 5 | LR: 0.001000
Train Loss: 0.3167 | Train F1: 0.2938
Val Loss: 0.3724 | Val F1: 0.2557
------------------------------
Epoch 6 | LR: 0.001000
Train Loss: 0.3091 | Train F1: 0.3341
Val Loss: 0.4351 | Val F1: 0.1934
------------------------------
Epoch 7 | LR: 0.001000
Train Loss: 0.3015 | Train F1: 0.3640
Val Loss: 0.4062 | Val F1: 0.2105
------------------------------
Epoch 8 | LR: 0.001000
Train Loss: 0.2945 | Train F1: 0.3883
Val Loss: 0.3827 | Val F1: 0.2529
------------------------------
Epoch 9 | LR: 0.000700
Train Loss: 0.2893 | Train F1: 0.3976
Val Loss: 0.3803 | Val F1: 0.2599
------------------------------
Epoch 10 | LR: 0.000700
Train Loss: 0.2785 | Train F1: 0.4337
Val Loss: 0.3722 | Val F1: 0.2350
------------------------------
Epoch 11 | LR: 0.000700
Train Loss: 0.2741 | Train F1: 0.4520
Val Loss: 0.3866 | Val F1: 0.2282
------------------------------
Epoch 12 | LR: 0.000700
Train Loss: 0.2698 | Train F1: 0.4665
Val Loss: 0.3617 | Val F1: 0.2714
------------------------------
Epoch 13 | LR: 0.000700
Train Loss: 0.2658 | Train F1: 0.4726
Val Loss: 0.3430 | Val F1: 0.3105
------------------------------
Epoch 14 | LR: 0.000700
Train Loss: 0.2608 | Train F1: 0.4937
Val Loss: 0.3632 | Val F1: 0.2925
------------------------------
Epoch 15 | LR: 0.000700
Train Loss: 0.2568 | Train F1: 0.4951
Val Loss: 0.3577 | Val F1: 0.3281
------------------------------ -->
